# 0. 项目简介

在车牌检测任务中，最简单的流程就是车牌检测+车牌识别两个步骤，但当镜头没有正对车牌的时候，图片中的车牌会有透视变形，增加识别任务的难度。

针对拍摄角度引起的透视变形，可再增加一步车牌校正的流程，整个任务流为：车牌检测、车牌校正、车牌识别

![](https://ai-studio-static-online.cdn.bcebos.com/f70c13cfbf494a3980b29619b85d13d0240d261c04cb441cbdc1f4a788426faf)

对于车牌检测部分，使用常用的检测算法yolo，可以输出目标的检测框和分类概率，但检测框还不能简化校正工作，若能识别出车牌的4个角点就能直接进行矫正了。

与yoloface一样，可在yolo框架中添加关键点回归分支，从而实现对车牌4个角点的检测

本项目在yolov5基础上添加了4个关键点的回归，实现了车牌和关键点的同时检测。yolov5来自于aistudio平台的开源项目[https://aistudio.baidu.com/aistudio/projectdetail/2580805](https://aistudio.baidu.com/aistudio/projectdetail/2580805) 感谢作者的工作。

最终项目的效果如下，关于后续车牌识别任务可移步：[https://aistudio.baidu.com/aistudio/projectdetail/5628649](https://aistudio.baidu.com/aistudio/projectdetail/5628649)

![](https://ai-studio-static-online.cdn.bcebos.com/f83218d932f641dd8e1ecdbcdc9caf8b9108d3904c8f49fa9e994f63eaab14ed)

本文1-6节为训练实践部分，7-8节为原理说明+代码解释。



# 1. 数据处理

数据集使用CCPD数据集，共计有9万多张图片，本项目为缩短训练时间仅使用其中2万张进行训练。

CCPD标注解释如下：

![](https://ai-studio-static-online.cdn.bcebos.com/60c819f0df2340eab90a8ce53150cdb315e7df5e6fe24ce7b1fda0bdc1784e2b)


解压数据集，处理为yolo格式，这里构造的label除了真实框还要加上车牌4个顶点的坐标，标签示意如下，注意都需要相对图片高宽进行归一化。

![](https://ai-studio-static-online.cdn.bcebos.com/7829230060f340ea83289fc2807810dc1785a6a3ad6a454d8c7154d419874b5e)



先下载数据集 https://aistudio.baidu.com/datasetdetail/17968

放到项目根目录 /LPD-project/CCPD2019.zip

然后解压 7z x CCPD2019.zip -oCCPD2019

下面就软连接到两个项目里面使用，比较方便

In [ ]:
!rm data
!ln -s ../assets/CCPD2019 data

In [1]:
# # 数据解压
# !unzip -o -q -d /home/aistudio/data /home/aistudio/data/data17968/CCPD2019.zip

In [ ]:
# 查看数据量
!cd data/ccpd_base && ls -l | grep "^-" | wc -l

In [ ]:
# 数据集转yolo格式
import json
import random
import cv2
import os
from tqdm import tqdm
import shutil


def xyxy2xywh(xyxy, img_size):
    img_h, img_w = img_size
    x1, y1, x2, y2 = xyxy
    x_c = (x1 + x2) / 2 / img_w
    y_c = (y1 + y2) / 2 / img_h
    h = (y2 - y1) / img_h
    w = (x2 - x1) / img_w
    return x_c, y_c, w, h


def parse_ccpd(path):
    """
    CCPD数据集的图片名称即是label:
    0152-4_14-224&551_398&624-388&610_224&624_234&565_398&551-0_0_30_27_31_9_31-97-108.jpg
          ^      ^       ^       ^       ^       ^       ^            ^         ^   ^
          |  框左上角  框右下角  右下角点 左下角点 左上角点  右上角点   车牌号码    亮度  模糊度
    水平/垂直倾角
    亮度数值越大，车牌越亮；模糊度数值越小，车牌越模糊。

    path: 图片文件夹
    """
    parse_list = []
    print('parse ccpd:')
    data_souce = os.listdir(path)
    random.seed(0)
    random.shuffle(data_souce)
    data_souce = data_souce[:30000]  # 少部分数据集进行测试先
    for filename in tqdm(data_souce):
        if "-" not in filename:  # 对于np等无标签的图片，过滤
            continue
        subname, landmarks = filename.split("-", 4)[2: 4]  # 获取真实框坐标,和车牌四角坐标
        # 过滤非图片数据
        extension = filename.split(".", 1)[1]
        if not extension == 'jpg':
            continue
        img = cv2.imread(os.path.join(path, filename))
        if img is None:  # 自动删除失效图片（下载过程有的图片会存在无法读取的情况）
            os.remove(os.path.join(path, filename))
            continue

        # 获取bounding box的x,y,h,w  coco格式
        lt, rb = subname.split("_", 1)
        lx, ly = lt.split("&", 1)  # 左上角坐标
        rx, ry = rb.split("&", 1)  # 右下角坐标

        points = landmarks.split('_')
        rb_x, rb_y = points[0].split('&')
        lb_x, lb_y = points[1].split('&')
        lt_x, lt_y = points[2].split('&')
        rt_x, rt_y = points[3].split('&')

        # ccpd数据集中每个图片只有1个检测框
        out_dic = {
            'img_name': filename, 
            'box': [int(lx), int(ly), int(rx), int(ry)],
            'img_size': img.shape[:2],  # h,w 
            'landmarks':[(int(rb_x), int(rb_y)), (int(lb_x),int(lb_y)), (int(lt_x),int(lt_y)), (int(rt_x),int(rt_y))]
        }
        parse_list.append(out_dic)
    print('len_data:', len(parse_list))
    return parse_list


def train_test_val_split_random(img_index, ratio_train=0.8, ratio_test=0.1, ratio_val=0.1):
    # 这里可以修改数据集划分的比例。
    assert int(ratio_train + ratio_test + ratio_val) == 1
    train_index = int(len(img_index) * ratio_train)
    val_index = int(len(img_index) * (ratio_train+ratio_val))
    train_img = img_index[:train_index]
    val_img = img_index[train_index:val_index]
    test_img = img_index[val_index:]
    print("NUMS of train:val:test = {}:{}:{}".format(len(train_img), len(val_img), len(test_img)))
    return train_img, val_img, test_img


def ccpd2yolo(root_path, imgDir, save_path):
    # classes = ['license Plate']
    print("Loading data from ", imgDir)
    data_source = parse_ccpd(imgDir)
    data_len = 20000  # 20000如果效果不好，可以使用全量数据
    index_list = list(range(len(data_source)))
    random.shuffle(index_list)
    indexes = index_list[:data_len]
    print('Total data: %s, needed: %s' % (len(data_source), data_len))

    print("random split...")
    train_img, val_img, test_img = train_test_val_split_random(indexes, 0.8, 0.1, 0.1)

    # 构建yolo数据集
    datas_index = [train_img, val_img, test_img]
    data_names = ['train', 'val', 'test']
    for i, img_index_list in enumerate(datas_index):
        img_folder = os.path.join(save_path, data_names[i] + '/images')
        label_folder = os.path.join(save_path, data_names[i] + '/labels')
        if not os.path.exists(img_folder):
            os.makedirs(img_folder)
        if not os.path.exists(label_folder):
            os.makedirs(label_folder)

        for i in img_index_list:
            item = data_source[i]
            img_name = item['img_name']
            img_box = item['box']
            img_size = item['img_size']  # h w
            r_bottom, l_bottom, l_top, r_top  = item['landmarks']  
            rb_x, rb_y = r_bottom[0] / img_size[1], r_bottom[1] / img_size[0]
            lb_x, lb_y = l_bottom[0] / img_size[1], l_bottom[1] / img_size[0]
            lt_x, lt_y = l_top[0] / img_size[1], l_top[1] / img_size[0]
            rt_x, rt_y = r_top[0] / img_size[1], r_top[1] / img_size[0]
            
            x_c, y_c, w, h = xyxy2xywh(img_box, img_size)
            # 分类  框中心x  框中心y  宽  高  右下x   右下y   左下x   左下y   左上x   左上y   右上x   右上y
            label = f'0 {x_c} {y_c} {w} {h} {rb_x} {rb_y} {lb_x} {lb_y} {lt_x} {lt_y} {rt_x} {rt_y}'
            
            img_file = os.path.join(imgDir, img_name)
            save_img = os.path.join(img_folder, img_name)
            shutil.copy(img_file, save_img)

            save_label = os.path.join(label_folder, img_name.replace('.jpg', '.txt'))
            with open(save_label, 'w', encoding='utf-8') as f:
                f.write(label)
            
root_dir = 'data'
img_dir = 'data/ccpd_base'
save_path = 'datasets'
if not os.path.exists(save_path):
    os.mkdir(save_path)

ccpd2yolo(root_dir, img_dir, save_path)

# 2. yolov5配置部分
配置数据集文件，其他暂时默认

这里车牌只算作一个分类，若遇到双层车牌可考虑添加一个分类，在任务中将识别到的双层车牌切割拼接成单层车牌的形式再进行识别。

In [ ]:
import os
yaml_content = f'''
path: {os.path.join(os.getcwd(), 'datasets')}  # dataset root dir
train: train
val: val  
test: test
# Classes
names:
  0: plate
'''
with open('yolov5-Paddle/data/plate.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

# 3. 训练
这里选择的是32G环境进行训练的，batchsize设置为了128，20000张数据集的训练10个epoch的时长约为1小时。

因为引入了关键点的回归，结果图信息不全，有需求的朋友可以自己修改下源码，这里影响不大。
![](https://ai-studio-static-online.cdn.bcebos.com/3b7e56e81810437d890b9c47b8ffe35781f65a519c0f4859b0eebf6e788d109a)


In [ ]:
# 开始训练，如有依赖包错误请重启内核后再次运行
!cd yolov5-Paddle at&& \
python train.py --data plate.yaml \
                --workers 2 \
                --img 640 \
                --epochs 10 \
                --cfg yolov5n.yaml \
                --weights yolov5n.pdparams \
                --batch-size 64 \
                # --cos-lr 

# 4. 验证

车牌检测的效果还是好的，下面是完美的PR曲线

![](https://ai-studio-static-online.cdn.bcebos.com/54795e1ee1be4f469f2d0708e86ca9dcb35fb9939cf240c0adaeef13f0081650)

对于关键点检测的效果这里没有用客观指标来评估，下面是几个验证样本的结果可视化图，关键点都检测了出来：

![](https://ai-studio-static-online.cdn.bcebos.com/7935e425eeec49dc90660b2942e5095477384d3d44ba4957a1c05cb64fe7ce71)


In [ ]:
# 验证
!cd yolov5-Paddle && \
python val.py --data plate.yaml --weights runs/train/exp3/weights/best.pdparams --img 640 --conf 0.5 --iou-thres 0.3

In [ ]:
# 推理
!cd yolov5-Paddle && \
python detect.py --weights ./runs/train/exp3/weights/best.pdparams --source ../test

# 5. 导出onnx


In [ ]:
# 更换自己想要导出为onnx的权重地址
!cd yolov5-Paddle && \
python export.py --weights runs/train/exp3/weights/best.pdparams --include onnx --img 640 --opset 11

# 6. 业务流测试（检测+矫正+识别）

虽然本项目重点是检测加矫正部分，但本节中使用了本人另一个项目中识别模型，将整个流程串联起来进行测试。

LPRNet车牌识别：[https://aistudio.baidu.com/aistudio/projectdetail/5628649](https://aistudio.baidu.com/aistudio/projectdetail/5628649)

由于关键点的检测会存在误差，有时候矫正的图片并不是完全方正的，为了更好结合识别流程，在训练车牌识别模型的时候建议对数据集增加一些仿射变换的数据增强方法。另外CCPD数据集很不均衡，建议训练车牌识别模型的时候再收集一些数据。

In [ ]:
# 安装onnxruntime
!pip install onnxruntime-gpu

# 或者安装CPU版本
# !pip install onnxruntime

In [ ]:

import onnxruntime
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from utils import pre_process, post_process, draw_boxes, draw_points, four_point_transform
%matplotlib inline

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']  # onnx GPU推理  需要onnxruntime-gpu
# providers = ['CPUExecutionProvider']  # onnx CPU推理

"""数据前处理"""
img_path = 'test/test3.jpg'
img_raw = cv2.imread(img_path)

img_data, scale, padd_data = pre_process(img_raw, img_size=640)  # RGB

"""车牌检测"""
# 加载 ONNX 模型生成推理用 sess，更换自己的onnx文件
onnx_path = "yolov5-Paddle/runs/train/exp/weights/best.onnx"
sess = onnxruntime.InferenceSession(onnx_path, providers=providers)

# 使用 ONNXRuntime 推理
ort_inputs = {sess.get_inputs()[0].name: img_data}
result = sess.run(None, ort_inputs)
result = np.array(result)  # (1, 1, 25200, 14)
result = np.squeeze(result)

# 后处理
confidence = 0.5
iou = 0.3
hw = img_raw.shape[:2]
boxes, landmarks, confs, classes = post_process(result, confidence, iou, scale, padd_data[0], padd_data[1], hw)

"""车牌矫正"""
plate_img = four_point_transform(img_raw, landmarks[0])

# 可视化
color = (255,0,255)
img = draw_boxes(img_raw, boxes, confs, classes, color, thickness=5)
img_out = draw_points(img, landmarks, (0,0,255), 10)

img_out = img_out[:, :, ::-1]
plt.subplot(121)
plt.imshow(img_out)
plt.axis('off')
plt.subplot(122)
plate_show = plate_img[..., ::-1]
plt.imshow(plate_show)
plt.axis('off')
plt.show()

"""车牌识别部分"""
CHARS = ['京', '沪', '津', '渝', '冀', '晋', '蒙', '辽', '吉', '黑',
         '苏', '浙', '皖', '闽', '赣', '鲁', '豫', '鄂', '湘', '粤',
         '桂', '琼', '川', '贵', '云', '藏', '陕', '甘', '青', '宁',
         '新',
         '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
         'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K',
         'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V',
         'W', 'X', 'Y', 'Z', 'I', 'O', '-'
         ]
def reprocess(pred):
    pred_data = pred[0]
    pred_label = np.argmax(pred_data, axis=0)
    no_repeat_blank_label = []
    pre_c = pred_label[0]
    if pre_c != len(CHARS) - 1:  # 非空白
        no_repeat_blank_label.append(pre_c)
    for c in pred_label:  # dropout repeate label and blank label
        if (pre_c == c) or (c == len(CHARS) - 1):
            if c == len(CHARS) - 1:
                pre_c = c
            continue
        no_repeat_blank_label.append(c)
        pre_c = c
    char_list = [CHARS[i] for i in no_repeat_blank_label]
    return ''.join(char_list)

# 归一化，预处理
img_data = plate_img[:,:,::-1]  # BGR to RGB
img_data = cv2.resize(img_data,(94, 24))
img_data = (img_data - 127.5) / 127.5  # 归一化
img_data = np.transpose(img_data, (2,0,1))  # HWC to CHW
img_data = np.expand_dims(img_data, 0)  # to BCHW
np_data = np.array(img_data, dtype=np.float32)

# 加载 ONNX 模型生成推理用 sess_recog
onnx_path = os.path.join(os.getcwd(), "../LPRNet-LPD-Keypoint/save_onnx/lprnet.onnx")
sess_recog = onnxruntime.InferenceSession(onnx_path, providers=providers)

# 使用 ONNXRuntime 推理
ort_inputs = {sess_recog.get_inputs()[0].name: np_data}

result, = sess_recog.run(None, ort_inputs)
plate_str = reprocess(result)
print('车牌号识别为：', plate_str)


In [ ]:
from utils import plot_one_box

img_path = 'test/test3.jpg'
img_raw = cv2.imread(img_path)

plot_one_box(boxes[0], img_raw, label=plate_str)

plate_show = img_raw[..., ::-1]
plt.figure(figsize=(10, 10))
plt.imshow(plate_show)
plt.axis('off')
plt.show()

# 7. 车牌关键点说明

本节对怎样添加关键点的回归进行说明，重点阐述模型的关键点输出与处理

## 7.1 关键点检测分支

首先在检测头中增加8个维度的输出用做关键点的预测，设计每个grid cell的输出如下图所示，依次是检测框中心点位置、检测框宽高、置信度、关键点位置、分类信息，本项目为单分类，所以输出共计有14个维度

![](https://ai-studio-static-online.cdn.bcebos.com/5cf9de2f88834c73aa3e45054e00fe2015171f648b3e4062a9111b0faa530cee)

如下图所示，假设车牌的检测框和分类由图中填充阴影的grid cell进行预测，那么车牌的4个角点也由该cell进行预测。不直接预测点的坐标，而是预测点到该cell(grid_x, grid_y)的x、y轴距离，并且为了统一尺度让关键点更容易被学习，与检测框的宽高预测类似，进一步将模型的关键点输出调整为预测点到cell的x、y轴距离与anchor宽高的比值，以图中车牌的右下角点为例，就是红色线段与蓝色anchor宽高的比值。

![](https://ai-studio-static-online.cdn.bcebos.com/644746d7354e440982ba24f536c10e6f1f1aedf2de6245e0b5fa7c660bee534c)

在yolov5中，检测框的宽高预测输出由以下公式计算得到，它将宽高都限制在了（0,4）范围内，即预测的宽高不超过anchor的4倍：

$(sigmoid(pred) * 2)^2$

这里也对模型的输出通过以下公式计算来作为关键点的预测输出，其值限制在了（-2,2）范围内，即关键点与cell坐标的差值不超过anchor的2倍。并且这样处理后模型输出的关键点也和宽高的输出是在同一尺度:

$(sigmoid(landmarks) * 2)^2 - 2$

在预测的时候用该比值乘以anchor再加上cell坐标就得到了关键点在图片中的坐标位置

## 7.2 损失函数

对关键点的回归采用的是WingLoss，当然也可以用其他损失函数比如SmoothL1Loss，这里对WingLoss进行一个简要的介绍：

wingloss公式如下：

![](https://ai-studio-static-online.cdn.bcebos.com/3217828087b44bf9bdecd9e764b06b8b254a3a02249e4a1b82adf400e79c40c5)

正数w将损失函数的非线性范围限制在了\[-w, w\]范围内，C是一个常数实现线性与非线性部分的平滑链接。

wingloss增强了小误差的影响，平衡了大误差和小误差的贡献，但注意ε不能设置的太小，否则对于小误差可能会梯度爆炸。

![](https://ai-studio-static-online.cdn.bcebos.com/df957b325faf448694f55ceea28be3bca82da674db5c41748b3e8f714eb83a08)


# 8. yolov5源码修改详情
本章节从代码层面对yolov5添加关键点回归分支进行说明

## 8.1 数据加载 + 增广
首先是数据标签的加载，需要在原来的基础上长度增加8，对dataloader.py中的LoadImagesAndLabels类的__getitem__方法进行适配。

**1. 归一化数据的还原，并且按照pad数据进行修改（mosaic）**
    
   在xywhn2xyxy方法中添加关键点的还原，labels[:, 1:] = xywhn2xyxy(labels[:, 1:], w, h, padw, padh)，将图片外的关键点置为-1
    
    ```
    def xywhn2xyxy(x, w=640, h=640, padw=0, padh=0):
        ...
        # 修改：将归一化的4点坐标还原, 并且按照mosaic的pad数据修改
        y[:, 4] = np.array(x[:, 4] > 0, dtype=np.int32) * (w * x[:, 4] + padw) + (np.array(x[:, 4] > 0, dtype=np.int32) - 1)
        y[:, 5] = np.array(x[:, 5] > 0, dtype=np.int32) * (h * x[:, 5] + padh) + (np.array(x[:, 5] > 0, dtype=np.int32) - 1)
        y[:, 6] = np.array(x[:, 6] > 0, dtype=np.int32) * (w * x[:, 6] + padw) + (np.array(x[:, 6] > 0, dtype=np.int32) - 1)
        y[:, 7] = np.array(x[:, 7] > 0, dtype=np.int32) * (h * x[:, 7] + padh) + (np.array(x[:, 7] > 0, dtype=np.int32) - 1)
        y[:, 8] = np.array(x[:, 8] > 0, dtype=np.int32) * (w * x[:, 8] + padw) + (np.array(x[:, 8] > 0, dtype=np.int32) - 1)
        y[:, 9] = np.array(x[:, 9] > 0, dtype=np.int32) * (h * x[:, 9] + padh) + (np.array(x[:, 9] > 0, dtype=np.int32) - 1)
        y[:, 10] = np.array(x[:, 10] > 0, dtype=np.int32) * (w * x[:, 10] + padw) + (np.array(x[:, 10] > 0, dtype=np.int32) - 1)
        y[:, 11] = np.array(x[:, 11] > 0, dtype=np.int32) * (h * x[:, 11] + padh) + (np.array(x[:, 11] > 0, dtype=np.int32) - 1)
        return y
    ```

**2. 其他数据增强的适配**
    
   random_perspective方法中标签的变换，需要将关键点标签也同时做变换，并且将超出图片的点置为-1
    
    ```
        ...
        xy = np.ones((n * 8, 3))
        # 修改：除了检测框的4个角点，添加车牌的4个角点，进行变换
        xy[:, :2] = targets[:, [1, 2, 3, 4, 1, 4, 3, 2, 5, 6, 7, 8, 9, 10, 11, 12]].reshape(n * 8, 2)  # x1y1, x2y2, x1y2, x2y1
        xy = xy @ M.T  # transform
        xy = (xy[:, :2] / xy[:, 2:3] if perspective else xy[:, :2]).reshape(n, 16)  # perspective rescale or affine
        ...
        landmarks = xy[:, [8, 9, 10, 11, 12, 13, 14,15]]
        mask = np.array(targets[:, 5:] > 0, dtype=np.int32)
        landmarks = landmarks * mask
        landmarks = landmarks + mask - 1
        ...
    ```
    
   在LoadImagesAndLabels类的__getitem__方法中对左右翻转的操作进行关键点的左右调换，不考虑车牌翻转的影响，都按右下角、左下角、左上角、右上角的顺序进行输出，一定程度降低学习难度
     
     ```
        labels[:, 5] = np.where(labels[:, 5] < 0, -1, 1 - labels[:, 5])
        labels[:, 7] = np.where(labels[:, 7] < 0, -1, 1 - labels[:, 7])
        labels[:, 9] = np.where(labels[:, 9] < 0, -1, 1 - labels[:, 9])
        labels[:, 11] = np.where(labels[:, 11] < 0, -1, 1 - labels[:, 11])
        #左右镜像后交换标签位置
        right_bottom = np.copy(labels[:, [5, 6]])
        left_top = np.copy(labels[:, [9, 10]])
        labels[:, [5, 6]] = labels[:, [7, 8]]
        labels[:, [7, 8]] = right_bottom
        labels[:, [9, 10]] = labels[:, [11, 12]]
        labels[:, [11, 12]] = left_top
     ```

## 8.2 检测头修改
**1. 对检测头进行修改，添加关键点分支**

   检测头初始化的时候增加8个维度的输出：
     
    ```
    class Detect(nn.Layer):
        def __init__(self, nc=80, anchors=(), ch=(), inplace=True):  # detection layer
            super().__init__()
            self.no = nc + 5 + 8  # number of outputs per anchor
            ...
    ```
    
**2. 预测时，检测头的前向计算需要按照第7节介绍的原理对输出进行处理：**
    
   这里修改的推理部分，在本yolov5框架中训练时的检测头输出则是在loss计算的时候处理
    
    ```
    if not self.training:  # inference
        ...
        else:  # Detect (boxes only)
            # 修改：这里的输出是：xy, wh, conf, landmarks, cls
            xy, wh, conf = F.sigmoid(x[i][:, :, :, :, 0:5]).split((2, 2, 1), 4)
            # 这里限制landmarks的范围为（-2,2）
            landmarks = (F.sigmoid(x[i][:, :, :, :, 5:13]) * 2)**2 - 2
            cls = F.sigmoid(x[i][:, :, :, :, 13:])

            xy = (xy * 2 + self.grid[i]) * self.stride[i]  # xy
            wh = (wh * 2) ** 2 * self.anchor_grid[i]  # wh

            # 因为xy回归的值是到grid点的距离2sigmoid(pred)-0.5, _make_grid中已经将grid减去了0.5, landmarks回归的值是关键点到grid距离与anchor的比值, 需要把0.5加上。
            landm1 = landmarks[:, :, :, :, 0:2] * self.anchor_grid[i] + (self.grid[i] + 0.5) * self.stride[i]  # landmark x1 y1
            landm2 = landmarks[:, :, :, :, 2:4] * self.anchor_grid[i] + (self.grid[i] + 0.5) * self.stride[i]  # landmark x2 y2
            landm3 = landmarks[:, :, :, :, 4:6] * self.anchor_grid[i] + (self.grid[i] + 0.5) * self.stride[i]  # landmark x3 y3
            landm4 = landmarks[:, :, :, :, 6:8] * self.anchor_grid[i] + (self.grid[i] + 0.5) * self.stride[i]  # landmark x4 y4
            y = paddle.concat((xy, wh, conf, landm1, landm2, landm3, landm4, cls), 4)
    ```

**3. 初始化修改**
    
   不管是单分类还是多分类，最终目标的置信度都是预测置信度乘以分类概率，而在单分类训练中又没有对分类进行学习，那么如何保证最后计算出的最终置信度是正确的呢？
    
   这个问题是通过初始化解决的，单分类时下面的初始化给bais赋值一个较大的正数，这样模型的分类输出经过sigmoid计算后是非常接近于1的，不会影响计算。

    ```
    def _initialize_biases(self, cf=None): 
        ...
            # 在原来基础上加8
            b[:, 5+8:5+8+m.nc] += math.log(0.6 / (m.nc - 0.99999)) if cf is None else paddle.log(cf / cf.sum())  # cls
    ```


## 8.3 Loss
接下来是Loss部分的相关修改

**1. 模型输出处理**

   这里loss回归的是第7节中提到的“预测点到cell的x、y轴距离与anchor宽高的比值”，所以与检测头中推理时的处理略有差异，没有加上cell坐标
   
    ```
    class ComputeLoss:
        ...
        def __call__(self, p, targets):
            ...
            lmark = paddle.zeros([1])  # lmark loss
            tcls, tbox, indices, anchors, tlandmarks, lmks_mask = self.build_targets(p, targets)  # targets
            ...
            # Losses
            for i, pi in enumerate(p): 
                ...
                pxy, pwh, _, plandmarks, pcls = pi[b, a, gj, gi].split((2, 2, 1, 8, self.nc), 1)
                ...
                plandmarks = (F.sigmoid(plandmarks)*2) ** 2 - 2.
                
                plandmarks[:, 0:2] = plandmarks[:, 0:2] * anchors[i]
                plandmarks[:, 2:4] = plandmarks[:, 2:4] * anchors[i]
                plandmarks[:, 4:6] = plandmarks[:, 4:6] * anchors[i]
                plandmarks[:, 6:8] = plandmarks[:, 6:8] * anchors[i]

                lmark += self.landmarks_loss(plandmarks, tlandmarks[i], lmks_mask[i])    
                ...
        def build_targets(self, p, targets):
            ...
            tcls, tbox, indices, anch, landmarks, lmks_mask = [], [], [], [], [], []
            ...
            # grid landmarks
            lks_mask = paddle.where(lks < 0, paddle.full_like(lks, 0.), paddle.full_like(lks, 1.0))
            lks[:, [0, 1]] = (lks[:, 0:2] - gij)
            lks[:, [2, 3]] = (lks[:, 2:4] - gij)
            lks[:, [4, 5]] = (lks[:, 4:6] - gij)
            lks[:, [6, 7]] = (lks[:, 6:8] - gij)
            ...
    ```
**2. 损失函数**

   损失函数为WingLoss，也可以选择SmoothL1Loss
    
    ```
    class WingLoss(nn.Layer):
        def __init__(self, w=10, e=2):
            super(WingLoss, self).__init__()
            # https://arxiv.org/pdf/1711.06753v4.pdf   Figure 5
            self.w = w
            self.e = e
            self.C = self.w - self.w * np.log(1 + self.w / self.e)

        def forward(self, x, t, sigma=1):
            abs_diff = (x - t).abs()
            flag = paddle.to_tensor((abs_diff.numpy() < self.w), dtype='float32')
            y = flag * self.w * paddle.log(1 + abs_diff / self.e) + (1 - flag) * (abs_diff - self.C)
            return y.sum()

    class LandmarksLoss(nn.Layer):
        def __init__(self, alpha=1.0):
            super(LandmarksLoss, self).__init__()
            self.loss_fcn = WingLoss()
            # self.loss_fcn = nn.SmoothL1Loss(reduction='sum')  # nn.SmoothL1Loss(reduction='sum')
            self.alpha = alpha

        def forward(self, pred, truel, mask):
            loss = self.loss_fcn(pred*mask, truel*mask)
            return loss / (paddle.sum(mask) + 10e-14)
    ```
    
   同时在超参数文件中添加关键点损失的增益系数：
    
    ```
    # yolov5-Paddle/data/hyps/hyp.scratch-low.yaml
    landmark: 0.01 # landmark loss gain
    ```





## 8.4 验证、日志

验证和日志部分的修改就不做详细介绍了，涉及以下几个部分

验证的损失等计算：yolov5-Paddle/val.py文件中run方法，nms相关部分；

输出图片：yolov5-Paddle/utils/plots.py文件中plot_images方法，添加关键点的可视化。

# 9. 总结

1. 本项目通过对yolov5的检测头添加关键点回归分支，实现了关键点检测的功能。

2. 利用关键点信息可对车牌进行矫正变换，是车牌识别任务的前部分流程。

3. 项目未对主干网络进行修改，后续性能提升工作可以考虑从此入手。

4. 在训练以及验证中除了landmarks loss以外，没有其他指标对关键点的检测效果进行评估，后续工作可进行完善。